In [1]:
from dotenv import load_dotenv
import nest_asyncio

nest_asyncio.apply()
load_dotenv()


True

# Generate Synthetic Test Dataset

In [4]:
from utils.utils import load_from_pickle

all_documents = load_from_pickle('all_documents')

In [5]:
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from utils.llm import get_llm, get_embedding_model

# generator with openai models
generator_llm = get_llm("gpt-4o-mini")
critic_llm = get_llm("gpt-4o-mini")
embeddings = get_embedding_model("text-embedding-3-small")

generator = TestsetGenerator.from_langchain(
    generator_llm,
    critic_llm,
    embeddings
)

# generate testset
testset = generator.generate_with_langchain_docs(all_documents, test_size=100, distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25})

Generating: 100%|██████████| 100/100 [34:24<00:00, 20.65s/it] 


In [6]:
from utils.utils import save_to_pickle

save_to_pickle(testset, 'testset')

Failed to batch ingest runs: LangSmithRateLimitError('Rate limit exceeded for https://api.smith.langchain.com/runs/batch. HTTPError(\'429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Monthly unique traces usage limit exceeded"}\')')
Failed to batch ingest runs: LangSmithRateLimitError('Rate limit exceeded for https://api.smith.langchain.com/runs/batch. HTTPError(\'429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Monthly unique traces usage limit exceeded"}\')')
Failed to batch ingest runs: LangSmithRateLimitError('Rate limit exceeded for https://api.smith.langchain.com/runs/batch. HTTPError(\'429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Monthly unique traces usage limit exceeded"}\')')


# Evaluation

In [2]:
import random
import string

def generate_id():
    return ''.join(random.choices(string.ascii_letters + string.digits, k=8)).lower()

In [3]:
from utils.utils import load_from_pickle

testset = load_from_pickle('testset')
dataset = testset.to_pandas()

c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,What challenges does the concept of kearifan l...,[DOC NAME: Undang-undang Nomor 15 Tahun 2023 T...,The concept of kearifan lokal in Bali faces ch...,simple,[{'title': 'Undang-undang Nomor 15 Tahun 2023 ...,True
1,What specific responsibilities does Lembaga Pe...,[DOC NAME: Undang-undang Nomor 4 Tahun 2023 Te...,Lembaga Penjamin Simpanan (LPS) has specific r...,simple,[{'title': 'Undang-undang Nomor 4 Tahun 2023 T...,True
2,What options do residents have to obtain addit...,[DOC NAME: Undang-undang Nomor 17 Tahun 2023 T...,Residents can obtain additional benefits throu...,simple,[{'title': 'Undang-undang Nomor 17 Tahun 2023 ...,True
3,What are the penalties and additional actions ...,[DOC NAME: Undang-undang Nomor 12 Tahun 2022 T...,According to Undang-undang Nomor 12 Tahun 2022...,simple,[{'title': 'Undang-undang Nomor 12 Tahun 2022 ...,True
4,What types of recognition can be awarded to in...,[DOC NAME: Undang-undang Nomor 11 Tahun 2022 T...,Individual athletes (Olahragawan) can be award...,simple,[{'title': 'Undang-undang Nomor 11 Tahun 2022 ...,True
...,...,...,...,...,...,...
92,What are the terminal facility requirements pe...,[DOC NAME: Undang-undang Nomor 6 Tahun 2023 Te...,Pembangunan Terminal harus dilengkapi dengan: ...,multi_context,[{'title': 'Undang-undang Nomor 6 Tahun 2023 T...,True
93,How does hak berdaulat shape Indonesia's conti...,[DOC NAME: Undang-undang Nomor 16 Tahun 2023 T...,The context does not provide specific informat...,multi_context,[{'title': 'Undang-undang Nomor 16 Tahun 2023 ...,True
94,What aspects of Indonesia's correctional syste...,[DOC NAME: Undang-undang Nomor 22 Tahun 2022 T...,The aspects of Indonesia's correctional system...,multi_context,[{'title': 'Undang-undang Nomor 22 Tahun 2022 ...,True
95,How does a public offering protect investors a...,[DOC NAME: Undang-undang Nomor 4 Tahun 2023 Te...,A public offering protects investors by requir...,multi_context,[{'title': 'Undang-undang Nomor 4 Tahun 2023 T...,True


In [5]:
from utils.rag.basic_agent import caller_with_sources

questions = []
ground_truths = []
answers = []
contexts = []

index = 0
for i, row in dataset.iterrows():
    print(f"Processing {index}/{len(dataset)}")
    if i >= 20:
        break

    question = row["question"]
    questions.append(question)
    ground_truths.append(row["ground_truth"])

    answer, sources = caller_with_sources(question, generate_id())
    answers.append(answer)

    content = []
    for source in sources:
        content.append(source.page_content)

    contexts.append(content)
    index += 1

Processing 0/97


c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(
c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(
c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\langchain_

Processing 1/97
Processing 2/97
Processing 3/97
Processing 4/97
Processing 5/97
Processing 6/97
Processing 7/97
Processing 8/97
Processing 9/97
Processing 10/97
Processing 11/97
Processing 12/97
Processing 13/97


Failed to batch ingest runs: LangSmithRateLimitError('Rate limit exceeded for https://api.smith.langchain.com/runs/batch. HTTPError(\'429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Monthly unique traces usage limit exceeded"}\')')
Failed to batch ingest runs: LangSmithConnectionError("Connection error caused failure to POST https://api.smith.langchain.com/runs/batch  in LangSmith API. Please confirm your internet connection.. ConnectionError(ProtocolError('Connection aborted.', ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None)))")


Processing 14/97
Processing 15/97
Processing 16/97
Processing 17/97
Processing 18/97
Processing 19/97
Processing 20/97


Failed to batch ingest runs: LangSmithRateLimitError('Rate limit exceeded for https://api.smith.langchain.com/runs/batch. HTTPError(\'429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/batch\', \'{"detail":"Monthly unique traces usage limit exceeded"}\')')
Failed to batch ingest runs: LangSmithConnectionError("Connection error caused failure to POST https://api.smith.langchain.com/runs/batch  in LangSmith API. Please confirm your internet connection.. ConnectionError(ProtocolError('Connection aborted.', ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None)))")
Failed to batch ingest runs: LangSmithConnectionError("Connection error caused failure to POST https://api.smith.langchain.com/runs/batch  in LangSmith API. Please confirm your internet connection.. ConnectionError(ProtocolError('Connection aborted.', ConnectionAbortedError(10053, 'An established connection was aborted by the s

In [6]:
from datasets import Dataset

d = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

final_dataset = Dataset.from_dict(d)

In [19]:
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall, answer_similarity, answer_correctness
from ragas.metrics.critique import harmfulness
from ragas import evaluate
from utils.llm import get_llm, get_embedding_model
from ragas.run_config import RunConfig


llm = get_llm(model="llama3.1")
embed_model = get_embedding_model(model="BAAI/bge-m3")


score = evaluate(
  final_dataset, 
  metrics=[context_recall],
  llm=llm, 
  embeddings=embed_model,
  # run_config=RunConfig(max_workers=2)
  )


Task exception was never retrieved
future: <Task finished name='Task-19059' coro=<as_completed.<locals>.sema_coro() done, defined at c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\ragas\executor.py:35> exception=KeyError('idle_for')>
Traceback (most recent call last):
  File "C:\Users\arkan\AppData\Local\Programs\Python\Python39\lib\asyncio\tasks.py", line 256, in __step
    result = coro.send(None)
  File "c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\ragas\executor.py", line 37, in sema_coro
    return await coro
  File "c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\ragas\executor.py", line 109, in wrapped_callable_async
    raise e
  File "c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\ragas\executor.py", line 104, in wrapped_callable_async
    result = await callable(*args, **kwargs)
  File "c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\ragas\metrics\

ExceptionInRunner: The runner thread which was running the jobs raised an exeception. Read the traceback above to debug it. You can also pass `raise_exceptions=False` incase you want to show only a warning message instead.

In [ ]:
for key,val in score.items():
    print(f"{key}: {val}")

faithfulness: 1.0
answer_relevancy: 0.9565339337032137
context_precision: 0.97875457874705
context_recall: 1.0
answer_similarity: 0.8773020796207043
answer_correctness: 0.969325519905176
harmfulness: 0.0
